# Votos e votações - Câmara/Brasil

O arquivo nacional de votos é lido uma única vez por ano e separado por UF. Os IDs encontrados em cada estado são usados para separar também o arquivo anual de votações.

Saída em cada estado:

- `votos_YYYY.csv`
- `votacoes_YYYY.csv`

A etapa não interpreta ausência. Ela guarda somente os registros de voto existentes na fonte.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import shutil
import tempfile
import time

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CATALOGO = "workspace"
SCHEMA = "pi_ii_bronze"
VOLUME = "camara"
ID_LEGISLATURA = 57
ANOS = [2023, 2024, 2025, 2026]

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

ROOT = Path(f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}")
# diretório temporário exclusivo desta execução
# evita conflito de permissão em retry/serverless
TMP = Path(tempfile.mkdtemp(prefix="pi_ii_bronze_camara_brasil_"))

# Se False, anos que já estiverem completos são reaproveitados.
FORCAR_RECOLETA = False

retry = Retry(
    total=6,
    connect=6,
    read=6,
    status=6,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)

session = requests.Session()
session.headers.update({
    "Accept": "*/*",
    "User-Agent": "PI-II-Univesp-Bronze-Camara-Brasil/1.0",
})
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

def pasta_uf(uf):
    return ROOT / uf.lower()

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def nome_normalizado(valor):
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())

def encontrar_coluna(df, candidatos, contem_todos=None):
    mapa = {nome_normalizado(c): c for c in df.columns}

    for nome in candidatos:
        chave = nome_normalizado(nome)
        if chave in mapa:
            return mapa[chave]

    if contem_todos:
        termos = [nome_normalizado(x) for x in contem_todos]
        for c in df.columns:
            nc = nome_normalizado(c)
            if all(t in nc for t in termos):
                return c

    return None

def baixar(url, destino, timeout=(30, 900)):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    parcial = destino.with_suffix(destino.suffix + ".part")
    if parcial.exists():
        parcial.unlink()

    print("Baixando:", url)

    with session.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()

        total = int(r.headers.get("content-length") or 0)
        recebido = 0
        ultimo_print = time.time()

        with parcial.open("wb") as f:
            for bloco in r.iter_content(chunk_size=1024 * 1024):
                if not bloco:
                    continue

                f.write(bloco)
                recebido += len(bloco)

                if time.time() - ultimo_print >= 5:
                    if total:
                        print(
                            f"  {recebido / 1024**2:.1f} MB / "
                            f"{total / 1024**2:.1f} MB"
                        )
                    else:
                        print(f"  {recebido / 1024**2:.1f} MB")
                    ultimo_print = time.time()

    parcial.replace(destino)
    print(f"Concluído: {destino.name} ({destino.stat().st_size / 1024**2:.1f} MB)")
    return destino

def ler_csv_chunks(path, chunksize=100_000):
    return pd.read_csv(
        path,
        sep=";",
        encoding="utf-8",
        dtype=str,
        chunksize=chunksize,
        keep_default_na=False,
        na_filter=False,
    )

def append_local(df, path, cabecalho):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        path,
        sep=";",
        index=False,
        encoding="utf-8",
        mode="a",
        header=cabecalho,
    )

def copiar_para_volume(origem, destino):
    origem = Path(origem)
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    with origem.open("rb") as src, destino.open("wb") as dst:
        shutil.copyfileobj(src, dst, length=1024 * 1024 * 8)

def criar_csv_vazio(path, colunas):
    pd.DataFrame(columns=colunas).to_csv(
        path,
        sep=";",
        index=False,
        encoding="utf-8",
    )

def limpar_tmp(prefixo):
    pasta = TMP / prefixo
    pasta.mkdir(parents=True, exist_ok=True)
    return pasta

def carregar_deputados_brasil():
    partes = []

    for uf in UFS:
        path = pasta_uf(uf) / "deputados.csv"
        if not path.exists():
            raise FileNotFoundError(
                f"{path} não existe. Execute primeiro o notebook 00."
            )

        df = pd.read_csv(
            path,
            sep=";",
            dtype=str,
            keep_default_na=False,
        )
        partes.append(df)

    deputados = pd.concat(partes, ignore_index=True).drop_duplicates()

    id_col = encontrar_coluna(
        deputados,
        ["id", "idDeputado", "deputado_id"],
    )
    uf_col = encontrar_coluna(
        deputados,
        ["siglaUf", "uf", "deputado_siglaUf"],
    )

    if not id_col or not uf_col:
        raise RuntimeError(
            "Não encontrei as colunas de ID/UF em deputados.csv."
        )

    mapa = (
        deputados[[id_col, uf_col]]
        .assign(
            **{
                id_col: deputados[id_col].astype(str).str.strip(),
                uf_col: deputados[uf_col].astype(str).str.strip().str.upper(),
            }
        )
        .drop_duplicates(subset=[id_col])
        .set_index(id_col)[uf_col]
        .to_dict()
    )

    return deputados, mapa

def ano_completo(arquivos):
    return all(Path(x).exists() for x in arquivos)

def salvar_manifesto(nome, payload):
    payload = dict(payload)
    payload["gerado_em_utc"] = utc_now()

    path = ROOT / f"_manifest_{nome}_brasil.json"
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)

    print("Manifesto:", path)

print("Bronze Câmara:", ROOT)
print("UFs:", len(UFS))
print("Anos:", ANOS)


In [ ]:
deputados, dep_para_uf = carregar_deputados_brasil()
print("Deputados no mapa:", len(dep_para_uf))

def preparar_tmp(pasta_tmp, prefixo, ano):
    return {
        uf: pasta_tmp / f"{prefixo}_{uf.lower()}_{ano}.csv"
        for uf in UFS
    }


In [ ]:
resumo = []

for ano in ANOS:
    print("\n" + "=" * 70)
    print("ANO", ano)

    esperados = []
    for uf in UFS:
        esperados += [
            pasta_uf(uf) / f"votos_{ano}.csv",
            pasta_uf(uf) / f"votacoes_{ano}.csv",
        ]

    if not FORCAR_RECOLETA and ano_completo(esperados):
        print("Ano já completo; pulando.")
        resumo.append({"ano": ano, "status": "ja_existia"})
        continue

    pasta_tmp = limpar_tmp(f"votacoes_{ano}")

    # votos
    url_votos = (
        "https://dadosabertos.camara.leg.br/arquivos/"
        f"votacoesVotos/csv/votacoesVotos-{ano}.csv"
    )
    csv_votos = pasta_tmp / f"votacoesVotos-{ano}.csv"
    baixar(url_votos, csv_votos)

    tmp_votos = preparar_tmp(pasta_tmp, "votos", ano)
    cont_votos = {uf: 0 for uf in UFS}
    header_votos = {uf: True for uf in UFS}
    ids_votacao_por_uf = {uf: set() for uf in UFS}
    colunas_votos = None

    for chunk in ler_csv_chunks(csv_votos):
        if colunas_votos is None:
            colunas_votos = list(chunk.columns)

        uf_col = encontrar_coluna(
            chunk,
            [
                "deputado_siglaUf",
                "deputado.siglaUf",
                "siglaUfDeputado",
            ],
            contem_todos=["deputado", "uf"],
        )

        id_dep_col = encontrar_coluna(
            chunk,
            [
                "deputado_id",
                "deputado.id",
                "idDeputado",
                "id_deputado",
            ],
            contem_todos=["deputado", "id"],
        )

        id_vot_col = encontrar_coluna(
            chunk,
            ["idVotacao", "votacao_id", "id_votacao"],
            contem_todos=["id", "votacao"],
        )

        if not id_vot_col:
            raise RuntimeError(
                "Não encontrei o ID da votação no arquivo de votos. "
                f"Colunas: {list(chunk.columns)}"
            )

        if uf_col:
            serie_uf = (
                chunk[uf_col]
                .astype(str)
                .str.strip()
                .str.upper()
            )
        elif id_dep_col:
            serie_uf = (
                chunk[id_dep_col]
                .astype(str)
                .str.strip()
                .map(dep_para_uf)
                .fillna("")
            )
        else:
            raise RuntimeError(
                "Não encontrei UF nem ID do deputado no arquivo de votos. "
                f"Colunas: {list(chunk.columns)}"
            )

        recorte = chunk.loc[serie_uf.isin(UFS)].copy()
        recorte["_uf_recorte"] = serie_uf.loc[recorte.index].values

        for uf, grupo in recorte.groupby("_uf_recorte"):
            grupo = grupo.drop(columns=["_uf_recorte"])

            append_local(
                grupo,
                tmp_votos[uf],
                cabecalho=header_votos[uf],
            )
            header_votos[uf] = False
            cont_votos[uf] += len(grupo)

            ids_votacao_por_uf[uf].update(
                grupo[id_vot_col].astype(str).str.strip()
            )

    for uf in UFS:
        destino = pasta_uf(uf) / f"votos_{ano}.csv"

        if tmp_votos[uf].exists():
            copiar_para_volume(tmp_votos[uf], destino)
        else:
            criar_csv_vazio(destino, colunas_votos or [])

    # votações
    url_votacoes = (
        "https://dadosabertos.camara.leg.br/arquivos/"
        f"votacoes/csv/votacoes-{ano}.csv"
    )
    csv_votacoes = pasta_tmp / f"votacoes-{ano}.csv"
    baixar(url_votacoes, csv_votacoes)

    tmp_votacoes = preparar_tmp(pasta_tmp, "votacoes", ano)
    cont_votacoes = {uf: 0 for uf in UFS}
    header_votacoes = {uf: True for uf in UFS}
    colunas_votacoes = None

    for chunk in ler_csv_chunks(csv_votacoes):
        if colunas_votacoes is None:
            colunas_votacoes = list(chunk.columns)

        id_col = encontrar_coluna(
            chunk,
            ["id", "idVotacao", "votacao_id"],
        )
        if not id_col:
            raise RuntimeError(
                "Não encontrei o ID no arquivo de votações. "
                f"Colunas: {list(chunk.columns)}"
            )

        ids_chunk = chunk[id_col].astype(str).str.strip()

        for uf in UFS:
            ids_uf = ids_votacao_por_uf[uf]
            if not ids_uf:
                continue

            grupo = chunk.loc[ids_chunk.isin(ids_uf)].copy()
            if grupo.empty:
                continue

            append_local(
                grupo,
                tmp_votacoes[uf],
                cabecalho=header_votacoes[uf],
            )
            header_votacoes[uf] = False
            cont_votacoes[uf] += len(grupo)

    for uf in UFS:
        destino = pasta_uf(uf) / f"votacoes_{ano}.csv"

        if tmp_votacoes[uf].exists():
            copiar_para_volume(tmp_votacoes[uf], destino)
        else:
            criar_csv_vazio(destino, colunas_votacoes or [])

        print(
            f"{uf}: votos={cont_votos[uf]:,} | "
            f"votacoes={cont_votacoes[uf]:,}"
        )

    resumo.append({
        "ano": ano,
        "status": "ok",
        "votos_por_uf": cont_votos,
        "votacoes_por_uf": cont_votacoes,
    })

salvar_manifesto("votacoes", {"resumo": resumo})

tabela = pd.DataFrame([
    {
        "ano": x["ano"],
        "status": x["status"],
        "votos_distribuidos": (
            sum(x.get("votos_por_uf", {}).values())
            if x.get("votos_por_uf") else None
        ),
        "votacoes_distribuidas": (
            sum(x.get("votacoes_por_uf", {}).values())
            if x.get("votacoes_por_uf") else None
        ),
    }
    for x in resumo
])

if "display" in globals():
    display(tabela)
else:
    print(tabela.to_string(index=False))
